# StockSense Pipeline Runner
**Run this notebook in Google Colab to populate the Neon database.**

----

## Before you run anything, read this doc first

### 1. Enable GPU (required for FinBERT in Cell 7)
- Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU** -> Save
- Do this BEFORE running any cells
- If you already started without GPU: Runtime -> Resart session, then re-run from the top

### 2. Set up Colab Secrets (required — do this once)
Click the key icon in the left sidebar, then add:

| Secret Name   | What to put there                                      |
|---------------|--------------------------------------------------------|
| `sshkey`     | Your GitHub SSH private key (`~/.ssh/id_ed25519`)      |
| `DATABASE_URL`| The Neon PostgreSQL connection string (get from Katie) |

Toggle Notebook access ON for both secrets.

Your SSH key must have read access to the SCCapstone/ai_roosters repo.

Ask Katie for the DATABASE_URL if you don't have it.

### 3. Run order
| Cell | What it does | Required? |
|------|-------------|-----------|
| 1 | Clone repo | Yes |
| 2 | Navigate to backend | Yes |
| 3 | Install dependencies | Yes |
| 3b | Verify GPU | Yes |
| 4 | Load secrets into env | Yes |
| 5 | Create DB tables | Yes (first time only) |
| 6 | Ingest stock price data | Yes |
| 7 | Run FinBERT on articles | Only after article CSV/HuggingFace is ready |
| 8 | Run sentiment aggregator | Only after Cell 7 |

### 4. Article data (Cell 7)
Cell 7 needs a CSV with columns: `published_at, title, description, url`
- Update `NEWS_CSV_PATH` in Cell 7 to point to the csv file
- OR replace the CSV loader with HuggingFace `load_dataset()` streaming

OR this can be changed depending on what Kevin's article pipeline determines

### 5. Never commit our secrets
Before saving this notebook to GitHub:
**Edit -> Clear all outputs** -> this removes any printed values from cells


In [ ]:
# ============================================================
# CELL 1: Clone private repo using SSH key from Colab Secrets
# ============================================================
import os
from google.colab import userdata

# Write private key to disk (fix Windows line endings)
os.makedirs("/root/.ssh", exist_ok=True)
key = userdata.get("sshkey")
key = key.replace("\r\n", "\n").replace("\r", "\n")
if not key.endswith("\n"):
    key += "\n"

with open("/root/.ssh/id_ed25519", "w") as f:
    f.write(key)
os.chmod("/root/.ssh/id_ed25519", 0o600)

# Add GitHub to known hosts (prevents interactive prompt)
!ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null

# Clone the repo
!git clone git@github.com:SCCapstone/ai_roosters.git

print("Clone complete.")


In [ ]:
# ============================================================
# CELL 2: Navigate into the backend folder
# ============================================================
%cd /content/ai_roosters/backend

# Verify in the right place
!pwd
!ls  # should show: app/, requirements-pipeline.txt, API.Dockerfile, etc.


In [ ]:
# Cell 3 - install system build tools first, then packages
!apt-get install -y --quiet libpq-dev gcc build-essential
#!pip install -r requirements-pipeline.txt --prefer-binary 2>&1 | tail -50
# Install core packages first (known-good binary wheels)
!pip install --prefer-binary \
    fastapi==0.95.2 uvicorn==0.22.0 sqlalchemy==1.4.41 \
    psycopg2-binary==2.9.6 "pydantic[email]==1.10.9" \
    python-jose[cryptography] "passlib[bcrypt]==1.7.4" bcrypt==4.0.1 \
    pandas numpy yfinance requests httpx python-dotenv \
    email-validator beautifulsoup4 tqdm \
    torch transformers xgboost scikit-learn datasets matplotlib

# Skip papermill install, it causes errors.
# Papermill is not needed for running the pipeline scripts

In [ ]:
# ============================================================
# CELL 4: Load secrets into environment variables
# DATABASE_URL is the Neon connection string - stored in
# Colab Secrets (ask Katie for URL)
# ============================================================
from google.colab import userdata
import os, sys

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

# add backend to Python path so imports work
sys.path.insert(0, '/content/ai_roosters/debug_ai_roosters/backend')

print("Environment ready.")
print(f"DATABASE_URL set: {'yes' if os.environ.get('DATABASE_URL') else 'NO , check the secret name'}")


In [ ]:
# ============================================================
# CELL 5: Create all tables in Neon (safe to re-run ,
# uses CREATE TABLE IF NOT EXISTS )
# ============================================================

from app.db_init import init_db

init_db()
print("Database tables initialized.")



In [ ]:
# ============================================================
# CELL 6: Download 5 years of price data from yfinance
# and store it in the Neon 'stocks' table.
#
# update_existing=False means: skip rows that already exist.
# Safe to re-run , won't create duplicates.
# ============================================================
from app.services.ingesting_pipelines.prices_ingest import PriceIngestor

ingestor = PriceIngestor(os.environ["DATABASE_URL"])

tickers = [
    "AAPL",        # Apple
    "TSLA",        # Tesla
    "MSFT",        # Microsoft
    "GOOGL",       # Alphabet / Google
    "AMZN",        # Amazon
    "META",        # Meta (Facebook)
    "NVDA",        # Nvidia
    "JPM",         # JPMorgan Chase
    "BP",          # BP
    "RELIANCE.NS", # Reliance Industries
]

ingestor.ingest_multiple_stocks(
    tickers=tickers,
    start_date="2020-01-01",
    end_date=None,
    period=None,
    update_existing=False,
)

print("Price ingestion complete.")






In [ ]:
# ============================================================
# CELL 7: Score articles with FinBERT and store results in
# the Neon 'articles' table.
#
# REQUIRES: A CSV file with columns:
#   published_at, title, description, url
#
# Set NEWS_CSV_PATH to the path of article CSV/JSON/whatever
# (Kevin, please replace this as you see fit)
# ============================================================

# --- EDIT THIS PATH/CODE FOR HUGGING FACE CONTENT ---
os.environ["NEWS_CSV_PATH"] = "/content/articles.csv"

from app.services.sentiment.article_processing import run_finbert_pipeline_from_env

result = run_finbert_pipeline_from_env()
print(f"FinBERT pipeline complete: {result}")


In [ ]:
# ============================================================
# CELL 8: Join articles + stocks to produce per-ticker
# daily sentiment summaries in 'sentiment_snapshots' table.
# Run AFTER Cell 6 (prices) AND Cell 7 (articles)complete.
# ============================================================
from app.services.sentiment.aggregator import run_sentiment_snapshot_pipeline_from_env

run_sentiment_snapshot_pipeline_from_env()
print("Sentiment aggregation complete.")
